In [ ]:
!pip install datasets pandas huggingface_hub

In [ ]:
from datasets import load_dataset, DatasetDict
import pandas as pd

reporting = load_dataset("JayShah07/multi_label_reporting")

print("Dataset structure:")
print(reporting)
print("\nFirst training example:")
print(reporting["train"][0])

In [ ]:
from datasets import concatenate_datasets

# Combine train and validation datasets
combined = concatenate_datasets([reporting["train"], reporting["validation"]])

# Convert to pandas DataFrame
df = combined.to_pandas()

# Rename user_query → query
df = df.rename(columns={"user_query": "query"})

# Identify label columns (all except query)
label_cols = [col for col in df.columns if col != "query"]

# Create a list of active labels for each row
df["labels"] = df.apply(lambda row: [col for col in label_cols if row[col] == 1], axis=1)

# OPTIONAL: Keep only query + labels
# df = df[["query", "labels"]]

# Save CSV
df.to_csv("reporting_combined.csv", index=False)

df.head()

In [ ]:
new_df = pd.read_csv("/content/synthetic_3500.csv")

new_df.head()

In [ ]:
import pandas as pd

# 1. Load both dataframes
df1 = pd.read_csv("/content/reporting_combined.csv")         # 383 rows approx
df2 = pd.read_csv("/content/synthetic_3500.csv")    # 3500 rows

# 2. Ensure df2 has the columns in the right order
desired_cols = [
    "query",
    "holdings",
    "capital_gains",
    "scheme_wise_returns",
    "investment_account_wise_returns",
    "portfolio_update",
    "Current Year",
    "Previous Year",
    "Daily",
    "Monthly",
    "Weekly",
    "Yearly",
    "None_date",
    "None_module",
    "labels"
]

# Reorder df2, moving "query" first and keeping labels last
df2 = df2[desired_cols]

# 3. Combine row-wise
combined_df = pd.concat([df1, df2], ignore_index=True)

# 4. Save final output
combined_df.to_csv("final_training_dataset.csv", index=False)

combined_df.head()

In [ ]:
import pandas as pd

df = pd.read_csv("final_training_dataset.csv")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

len(train_df), len(val_df), len(test_df)

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset_dict

In [ ]:
dataset_dict = dataset_dict.remove_columns(["__index_level_0__"])
dataset_dict

In [ ]:
from huggingface_hub import login
login()

In [ ]:
dataset_dict.push_to_hub("JayShah07/reporting_final_dataset")